In [1]:
from pathlib import Path
import sys

def find_project_root():
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / "utils").exists() and (p / "models").exists():
            return p
    return here

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

In [2]:
import torch
import numpy as np
import torch.nn as nn
import pandas as pd

from utils.reproducibility_utils import seed_everything
from utils.data_loading import load_data, load_test_data

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


## 🔧 Core Components (Proven)

In [3]:
class RNN(nn.Module):
    """Vanilla RNN - Proven best architecture"""
    def __init__(self, input_size, rnn_units=64, fc_units=64):
        super(RNN, self).__init__()
        self.rnn_layer = nn.RNN(input_size=input_size, hidden_size=rnn_units, 
                                num_layers=1, nonlinearity='relu', batch_first=True)
        self.fc = nn.Linear(rnn_units, fc_units)
        self.output_layer = nn.Linear(fc_units, 1)
        self.activation = nn.ReLU()
 
    def forward(self, x):
        x, _ = self.rnn_layer(x)
        x = x[:, -1, :]
        x = self.activation(self.fc(x))
        x = self.output_layer(x)
        return x


class FewShotWeightedL1Loss(nn.Module):
    """Weight = base + slope * max(0, |y| - threshold)"""
    def __init__(self, base_weight=1.0, slope=1.0, threshold=2.0):
        super().__init__()
        self.base_weight = base_weight
        self.slope = slope
        self.threshold = threshold
    
    def forward(self, pred, target):
        excess = torch.clamp(torch.abs(target) - self.threshold, min=0)
        weights = self.base_weight + self.slope * excess
        loss = torch.abs(pred - target) * weights
        return loss.mean()


def predict_with_model(model, test_loader, device='cuda'):
    model.eval()
    predictions = []
    with torch.no_grad():
        for x, _ in test_loader:
            pred = model(x.to(device))
            predictions.append(pred.cpu().numpy())
    return np.concatenate(predictions).flatten()


print("✅ Core components defined")

✅ Core components defined


In [4]:
# Load data
print("Loading data...")
train_loader, val_loader, pipeline = load_data(
    train_path="../datasets_challenge/train",
    train_ratio=0.98,
    batchsize=64,
    columns_to_drop=["sideSlip", "vyCG", "time", "longitude", "latitude"],
    sequence_length=20,
    threshold=True,
    threshold_value=3.0,
    threshold_column="vxCG",
    target="sideSlip",
    scaling=True,
    seed=42,
)

test_loader = load_test_data(
    test_path="../datasets_challenge/test",
    batchsize=512,
    pipeline=pipeline,
    load_anonymized_data=True
)

X_sample, _ = next(iter(train_loader))
input_size = X_sample.shape[2]
print(f"✅ Data loaded | Input size: {input_size}")

Loading data...
../datasets_challenge/train/train_part_001.csv
../datasets_challenge/train/train_part_002.csv
../datasets_challenge/train/train_part_003.csv
../datasets_challenge/train/train_part_004.csv
../datasets_challenge/train/train_part_005.csv
../datasets_challenge/train/train_part_006.csv
../datasets_challenge/train/train_part_004.csv
../datasets_challenge/train/train_part_005.csv
../datasets_challenge/train/train_part_006.csv
../datasets_challenge/train/train_part_007.csv
../datasets_challenge/train/train_part_007.csv
../datasets_challenge/train/train_part_008.csv
../datasets_challenge/train/train_part_009.csv
../datasets_challenge/train/train_part_008.csv
../datasets_challenge/train/train_part_009.csv
../datasets_challenge/train/train_part_010.csv
../datasets_challenge/train/train_part_011.csv
../datasets_challenge/train/train_part_010.csv
../datasets_challenge/train/train_part_011.csv
../datasets_challenge/train/train_part_012.csv
../datasets_challenge/train/train_part_013.c

## 🚀 Winning Two-Stage Training Function

In [5]:
def train_two_stage_aggressive(model, train_loader, val_loader, device='cuda',
                                stage1_epochs=30, stage2_epochs=30,
                                stage2_slope=1.0, stage2_threshold=2.0,
                                stage2_lr=0.00005, verbose=True):
    """
    Winning two-stage training:
    - Stage 1: Standard L1 (30 epochs, lr=0.0003)
    - Stage 2: Aggressive weighted loss (configurable)
    """
    
    # ========== STAGE 1: Standard L1 ==========
    if verbose:
        print(f"📍 Stage 1: L1 Loss, {stage1_epochs} epochs, lr=0.0003")
    
    criterion_s1 = nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)
    
    best_loss, best_state = np.inf, None
    for epoch in range(stage1_epochs):
        model.train()
        for x, y in train_loader:
            X, y = x.to(device), y.to(device).reshape(-1, 1)
            optimizer.zero_grad()
            loss = criterion_s1(model(X), y)
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_loss = sum(criterion_s1(model(x.to(device)), y.to(device).reshape(-1,1)).item() 
                       for x, y in val_loader) / len(val_loader)
        
        if val_loss < best_loss:
            best_loss, best_state = val_loss, model.state_dict().copy()
        
        if verbose and (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:2d} | Val: {val_loss:.5f}")
    
    model.load_state_dict(best_state)
    if verbose:
        print(f"  ✅ Stage 1 done | Best: {best_loss:.5f}")
    
    # ========== STAGE 2: Aggressive Weighted ==========
    if verbose:
        print(f"📍 Stage 2: slope={stage2_slope}, thresh={stage2_threshold}, lr={stage2_lr}, {stage2_epochs} epochs")
    
    criterion_s2 = FewShotWeightedL1Loss(base_weight=1.0, slope=stage2_slope, threshold=stage2_threshold)
    optimizer = torch.optim.Adam(model.parameters(), lr=stage2_lr)
    
    best_loss_s2, best_state_s2 = np.inf, None
    for epoch in range(stage2_epochs):
        model.train()
        for x, y in train_loader:
            X, y = x.to(device), y.to(device).reshape(-1, 1)
            optimizer.zero_grad()
            loss = criterion_s2(model(X), y)
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_loss = sum(nn.L1Loss()(model(x.to(device)), y.to(device).reshape(-1,1)).item() 
                       for x, y in val_loader) / len(val_loader)
        
        if val_loss < best_loss_s2:
            best_loss_s2, best_state_s2 = val_loss, model.state_dict().copy()
        
        if verbose and (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:2d} | Val (L1): {val_loss:.5f}")
    
    model.load_state_dict(best_state_s2)
    if verbose:
        print(f"  ✅ Stage 2 done | Best: {best_loss_s2:.5f}")
    
    return model, best_loss_s2


def train_baseline(model, train_loader, val_loader, device='cuda', num_epochs=50, verbose=True):
    """Standard L1 training (for baseline ensemble)"""
    if verbose:
        print(f"📍 Baseline: L1 Loss, {num_epochs} epochs, lr=0.0003")
    
    criterion = nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)
    
    best_loss, best_state, patience = np.inf, None, 0
    for epoch in range(num_epochs):
        model.train()
        for x, y in train_loader:
            X, y = x.to(device), y.to(device).reshape(-1, 1)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_loss = sum(criterion(model(x.to(device)), y.to(device).reshape(-1,1)).item() 
                       for x, y in val_loader) / len(val_loader)
        
        if val_loss < best_loss:
            best_loss, best_state, patience = val_loss, model.state_dict().copy(), 0
        else:
            patience += 1
            if patience > 10:
                break
        
        if verbose and (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:2d} | Val: {val_loss:.5f}")
    
    model.load_state_dict(best_state)
    if verbose:
        print(f"  ✅ Done | Best: {best_loss:.5f}")
    
    return model, best_loss


print("✅ Training functions defined")

✅ Training functions defined


## 🎯 Approach B: Vary Slope & Threshold (Different from task7_focused)

**task7_focused.ipynb** tests: LR & epochs (fixed slope=1.0, thresh=2.0)

**This notebook** tests: Slope & threshold variations

Current best: slope=1.0, threshold=2.0 → 0.21957

Test if slightly different slope/threshold improves few-shot further:

In [6]:
# ============================================================================
# APPROACH B: Vary slope & threshold (complementary to task7_focused)
# FIXED: lr=0.00005, epochs=30, seeds=[42, 2024], blend=50/50
# ============================================================================

CONFIGS = [
    # (name, slope, threshold) - lr=0.00005, 30ep fixed
    ("slope1.2_thresh1.8", 1.2, 1.8),   # Slightly more aggressive
    ("slope0.8_thresh2.2", 0.8, 2.2),   # Slightly less aggressive  
    ("slope1.0_thresh1.5", 1.0, 1.5),   # Lower threshold only
    ("slope1.3_thresh2.0", 1.3, 2.0),   # Higher slope only
]

# FIXED parameters (proven)
SEEDS = [42, 2024]
S2_LR = 0.00005
S2_EPOCHS = 30

results = {}

print("="*60)
print("🎯 APPROACH B: Slope & Threshold Variations")
print("   Fixed: lr=0.00005, epochs=30, seeds=[42,2024], blend=50/50")
print("="*60)

🎯 APPROACH B: Slope & Threshold Variations
   Fixed: lr=0.00005, epochs=30, seeds=[42,2024], blend=50/50


In [7]:
# Train baseline models (FIXED - 2 seeds only)
print("\n" + "="*60)
print("📦 BASELINE MODELS (2 seeds)")
print("="*60)

baseline_preds = []
for seed in SEEDS:
    print(f"\n🌱 Seed {seed}")
    seed_everything(seed)
    model = RNN(input_size=input_size).to(device)
    model, _ = train_baseline(model, train_loader, val_loader, device, verbose=True)
    baseline_preds.append(predict_with_model(model, test_loader, device))

baseline_ensemble = np.mean(baseline_preds, axis=0)
print(f"\n✅ Baseline ready")


📦 BASELINE MODELS (2 seeds)

🌱 Seed 42
📍 Baseline: L1 Loss, 50 epochs, lr=0.0003
📍 Baseline: L1 Loss, 50 epochs, lr=0.0003
  Epoch 10 | Val: 0.15430
  Epoch 10 | Val: 0.15430
  Epoch 20 | Val: 0.14026
  Epoch 20 | Val: 0.14026
  Epoch 30 | Val: 0.12778
  Epoch 30 | Val: 0.12778
  Epoch 40 | Val: 0.12864
  Epoch 40 | Val: 0.12864
  Epoch 50 | Val: 0.13112
  ✅ Done | Best: 0.11366
  Epoch 50 | Val: 0.13112
  ✅ Done | Best: 0.11366

🌱 Seed 2024
📍 Baseline: L1 Loss, 50 epochs, lr=0.0003

🌱 Seed 2024
📍 Baseline: L1 Loss, 50 epochs, lr=0.0003
  Epoch 10 | Val: 0.14568
  Epoch 10 | Val: 0.14568
  Epoch 20 | Val: 0.13979
  Epoch 20 | Val: 0.13979
  Epoch 30 | Val: 0.12182
  Epoch 30 | Val: 0.12182
  Epoch 40 | Val: 0.12261
  Epoch 40 | Val: 0.12261
  Epoch 50 | Val: 0.11564
  ✅ Done | Best: 0.11157
  Epoch 50 | Val: 0.11564
  ✅ Done | Best: 0.11157

✅ Baseline ready

✅ Baseline ready


In [8]:
# Train two-stage with different slope/threshold
for name, slope, threshold in CONFIGS:
    print(f"\n{'='*60}")
    print(f"📦 {name} (slope={slope}, threshold={threshold})")
    print("="*60)
    
    config_preds = []
    for seed in SEEDS:
        print(f"\n🌱 Seed {seed}")
        seed_everything(seed)
        model = RNN(input_size=input_size).to(device)
        
        model, _ = train_two_stage_aggressive(
            model, train_loader, val_loader, device=device,
            stage1_epochs=30, stage2_epochs=S2_EPOCHS,
            stage2_slope=slope, stage2_threshold=threshold, stage2_lr=S2_LR,
            verbose=True
        )
        config_preds.append(predict_with_model(model, test_loader, device))
    
    results[name] = np.mean(config_preds, axis=0)
    print(f"\n✅ {name} done")


📦 slope1.2_thresh1.8 (slope=1.2, threshold=1.8)

🌱 Seed 42
📍 Stage 1: L1 Loss, 30 epochs, lr=0.0003
  Epoch 10 | Val: 0.14362
  Epoch 10 | Val: 0.14362
  Epoch 20 | Val: 0.13699
  Epoch 20 | Val: 0.13699
  Epoch 30 | Val: 0.13157
  ✅ Stage 1 done | Best: 0.12387
📍 Stage 2: slope=1.2, thresh=1.8, lr=5e-05, 30 epochs
  Epoch 30 | Val: 0.13157
  ✅ Stage 1 done | Best: 0.12387
📍 Stage 2: slope=1.2, thresh=1.8, lr=5e-05, 30 epochs
  Epoch 10 | Val (L1): 0.12070
  Epoch 10 | Val (L1): 0.12070
  Epoch 20 | Val (L1): 0.11436
  Epoch 20 | Val (L1): 0.11436
  Epoch 30 | Val (L1): 0.11373
  ✅ Stage 2 done | Best: 0.11373
  Epoch 30 | Val (L1): 0.11373
  ✅ Stage 2 done | Best: 0.11373

🌱 Seed 2024
📍 Stage 1: L1 Loss, 30 epochs, lr=0.0003

🌱 Seed 2024
📍 Stage 1: L1 Loss, 30 epochs, lr=0.0003
  Epoch 10 | Val: 0.15741
  Epoch 10 | Val: 0.15741
  Epoch 20 | Val: 0.12909
  Epoch 20 | Val: 0.12909
  Epoch 30 | Val: 0.12075
  ✅ Stage 1 done | Best: 0.12075
📍 Stage 2: slope=1.2, thresh=1.8, lr=5e-05, 30

In [9]:
# Save 50/50 blends only (proven optimal ratio)
print("\n" + "="*60)
print("💾 SAVING PREDICTIONS (50/50 blend)")
print("="*60)

for name, ensemble in results.items():
    blend = (baseline_ensemble + ensemble) / 2
    filename = f"pred_v7b_{name}.csv"
    pd.DataFrame({"id": range(len(blend)), "predicted_sideslip": blend}).to_csv(filename, index=False)
    print(f"✅ {filename}")


💾 SAVING PREDICTIONS (50/50 blend)
✅ pred_v7b_slope1.2_thresh1.8.csv
✅ pred_v7b_slope0.8_thresh2.2.csv
✅ pred_v7b_slope1.0_thresh1.5.csv
✅ pred_v7b_slope1.3_thresh2.0.csv
✅ pred_v7b_slope1.3_thresh2.0.csv


In [10]:
# ============================================================================
# SUMMARY
# ============================================================================

print("\n" + "="*60)
print("📋 APPROACH B: Slope & Threshold Variations")
print("="*60)
print("\n🏆 Current Best: 0.21957 (slope=1.0, thresh=2.0)")
print("\n📁 Files to submit (4 submissions):")
for name, slope, threshold in CONFIGS:
    print(f"  - pred_v7b_{name}.csv  (slope={slope}, thresh={threshold})")

print("\n" + "="*60)
print("📋 APPROACH A (task7_focused.ipynb): LR & Epochs")
print("="*60)
print("  - pred_v7_lr3e5_40ep.csv")
print("  - pred_v7_lr5e5_40ep.csv")
print("  - pred_v7_lr2e5_50ep.csv")
print("  - pred_v7_lr3e5_35ep.csv")

print("\n💡 Total: 8 focused submissions across both approaches")


📋 APPROACH B: Slope & Threshold Variations

🏆 Current Best: 0.21957 (slope=1.0, thresh=2.0)

📁 Files to submit (4 submissions):
  - pred_v7b_slope1.2_thresh1.8.csv  (slope=1.2, thresh=1.8)
  - pred_v7b_slope0.8_thresh2.2.csv  (slope=0.8, thresh=2.2)
  - pred_v7b_slope1.0_thresh1.5.csv  (slope=1.0, thresh=1.5)
  - pred_v7b_slope1.3_thresh2.0.csv  (slope=1.3, thresh=2.0)

📋 APPROACH A (task7_focused.ipynb): LR & Epochs
  - pred_v7_lr3e5_40ep.csv
  - pred_v7_lr5e5_40ep.csv
  - pred_v7_lr2e5_50ep.csv
  - pred_v7_lr3e5_35ep.csv

💡 Total: 8 focused submissions across both approaches
